# Phase 4 — Cross-Variable Exploration

Pairplots, scatter vs time-to-event, vintage analysis, perforation depth vs Sw,
and correlation matrices.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
elif 'mari_poc' not in os.getcwd(): os.chdir('mari_poc')
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from scipy import stats as sp_stats

from src.config import (PROCESSED_DIR, FIGURES_DIR, COLOR_WET, COLOR_DRY,
                         COLOR_FORECAST, COLOR_EXCLUDED, HORIZONTAL_WELLS,
                         FORECAST_TARGETS, EXCLUDED_WELLS)
from src.features import detect_breakthrough, compute_wgr

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

panel = pd.read_parquet(PROCESSED_DIR / 'panel_long.parquet')
static = pd.read_parquet(PROCESSED_DIR / 'well_static.parquet')

# Get breakthrough events
panel_wgr = compute_wgr(panel)
bt_df = detect_breakthrough(panel_wgr)

# Merge static with breakthrough info
df = static.merge(bt_df, on='well', how='left')

# Compute time-to-event for survival: months from first_prod to breakthrough or censoring
df['tte_months'] = df.apply(
    lambda r: r['bt_month_index'] if r['bt_detected'] else r['production_months'],
    axis=1
).astype(float)
df['event'] = df['bt_detected'].astype(int)

# Perf midpoint
df['perf_midpoint_md'] = (df['top_perf_md'] + df['bottom_perf_md']) / 2

# Category for coloring
def well_category(well):
    if well in EXCLUDED_WELLS: return 'Excluded'
    if well in FORECAST_TARGETS: return 'Forecast'
    if well in HORIZONTAL_WELLS: return 'Horizontal'
    return 'Vertical'
df['category'] = df['well'].apply(well_category)

# Category for wet/dry coloring (among non-excluded, non-forecast)
def well_status(row):
    if row['well'] in EXCLUDED_WELLS: return 'Excluded'
    if row['well'] in FORECAST_TARGETS: return 'Forecast'
    if row['bt_detected']: return 'Wet'
    return 'Dry'
df['status'] = df.apply(well_status, axis=1)

print(f'Dataset: {len(df)} wells')
print(df[['well', 'status', 'bt_detected', 'bt_month_index', 'tte_months', 'event']].to_string())

Dataset: 22 wells
          well    status  bt_detected bt_month_index  tte_months  event
0     M-11-HRL       Wet         True            409       409.0      1
1   M-122H-HRL       Wet         True             12        12.0      1
2   M-123H-HRL  Forecast        False           <NA>        20.0      0
3   M-124H-HRL  Forecast        False           <NA>        19.0      0
4   M-125H-HRL  Forecast        False           <NA>        10.0      0
5   M-126H-HRL  Forecast        False           <NA>         9.0      0
6     M-13-HRL       Dry        False           <NA>       563.0      0
7     M-22-HRL       Dry        False           <NA>       525.0      0
8     M-41-HRL       Wet         True            329       329.0      1
9     M-50-HRL       Wet         True            379       379.0      1
10    M-51-HRL  Excluded         True            187       187.0      1
11    M-56-HRL       Wet         True            299       299.0      1
12    M-57-HRL       Dry        False         

## 1. Pairplot of Static Properties

In [2]:
# Pairplot for verticals only (horizontals have incomparable net_pay)
feats = ['porosity', 'permeability_md', 'sw', 'net_pay_m', 'perf_midpoint_md', 'chlorides_ppm']

# Filter to verticals + horizontals for pair plot
df_pair = df[~df['well'].isin(EXCLUDED_WELLS)].copy()

color_map = {'Wet': COLOR_WET, 'Dry': COLOR_DRY, 'Forecast': COLOR_FORECAST, 'Horizontal': '#9467bd'}

# For verticals, use full feature set; note horizontals separately
df_vert = df_pair[~df_pair['is_horizontal']].copy()

g = sns.pairplot(df_vert, vars=feats, hue='status',
                 palette={'Wet': COLOR_WET, 'Dry': COLOR_DRY},
                 diag_kind='hist', plot_kws={'alpha': 0.7, 's': 60})
g.fig.suptitle('Pairplot — Vertical Wells Only (Static Properties)', y=1.02)
g.savefig(FIGURES_DIR / '04_pairplot_verticals.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved 04_pairplot_verticals.png')

Saved 04_pairplot_verticals.png


## 2. Static Features vs Time-to-Event

In [3]:
# Scatter of each static feature vs TTE, with wells labelled
feats_tte = ['porosity', 'permeability_md', 'sw', 'net_pay_m', 'perf_midpoint_md', 'chlorides_ppm', 'gas_gravity']

# Only use wells with defined TTE (exclude forecast targets with very short histories)
df_tte = df[~df['well'].isin(FORECAST_TARGETS) & ~df['well'].isin(EXCLUDED_WELLS)].copy()

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, feat in enumerate(feats_tte):
    ax = axes[i]
    for _, row in df_tte.iterrows():
        color = COLOR_WET if row['bt_detected'] else COLOR_DRY
        if row['is_horizontal']: color = '#9467bd'
        ax.scatter(row[feat], row['tte_months'], c=color, s=60, zorder=3)
        label = row['well'].replace('M-','').replace('-HRL','')
        ax.annotate(label, (row[feat], row['tte_months']), fontsize=6,
                   xytext=(4, 4), textcoords='offset points')
    
    ax.set_xlabel(feat)
    ax.set_ylabel('Time to event (months)')
    ax.set_title(f'{feat} vs TTE')
    ax.grid(alpha=0.3)
    
    # Compute Spearman correlation
    valid = df_tte[[feat, 'tte_months']].dropna()
    if len(valid) >= 5:
        rho, p = sp_stats.spearmanr(valid[feat], valid['tte_months'])
        ax.text(0.05, 0.95, f'ρ={rho:.2f}, p={p:.3f}', transform=ax.transAxes,
               fontsize=8, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Remove extra axes
for j in range(len(feats_tte), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_features_vs_tte.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 04_features_vs_tte.png')

Saved 04_features_vs_tte.png


## Sw Wrong Sign Check

Expected physics: higher Sw → closer to GWC → faster breakthrough.
But if wells with LOW Sw broke through faster, that's a vintage confounder —
older wells (lower Sw from being higher in column) had more time to experience
depletion-driven water rise.

In [4]:
# Sw vs TTE analysis
df_vert_tte = df_tte[~df_tte['is_horizontal']].copy()

print('Sw vs TTE for vertical wells:')
for _, row in df_vert_tte.sort_values('sw').iterrows():
    status = 'WET' if row['bt_detected'] else 'DRY'
    print(f"  {row['well']:<15} Sw={row['sw']:.2f}  TTE={row['tte_months']:.0f}  {status}")

valid = df_vert_tte[['sw', 'tte_months']].dropna()
rho, p = sp_stats.spearmanr(valid['sw'], valid['tte_months'])
print(f'\nSpearman ρ(Sw, TTE) = {rho:.3f}, p = {p:.4f}')

if rho > 0:
    print('\n→ POSITIVE correlation: higher Sw → longer TTE.')
    print('  This is the WRONG physics sign. Higher Sw should mean FASTER breakthrough.')
    print('  Interpretation: likely a VINTAGE CONFOUNDER. Older wells drilled higher in the')
    print('  gas column (lower Sw) have had more time for depletion-driven water rise,')
    print('  while newer wells drilled near GWC (higher Sw) havent produced long enough.')
else:
    print('\n→ NEGATIVE correlation: higher Sw → shorter TTE. Matches physics expectation.')

Sw vs TTE for vertical wells:
  M-41-HRL        Sw=0.13  TTE=329  WET
  M-65-HRL        Sw=0.19  TTE=259  WET
  M-63-HRL        Sw=0.21  TTE=275  WET
  M-67-HRL        Sw=0.26  TTE=45  WET
  M-57-HRL        Sw=0.28  TTE=384  DRY
  M-56-HRL        Sw=0.30  TTE=299  WET
  M-13-HRL        Sw=0.31  TTE=563  DRY
  M-61-HRL        Sw=0.31  TTE=284  WET
  M-75-HRL        Sw=0.35  TTE=127  WET
  M-81-HRL        Sw=0.35  TTE=57  WET
  M-82-HRL        Sw=0.35  TTE=48  WET
  M-58-HRL        Sw=0.36  TTE=88  WET
  M-22-HRL        Sw=0.42  TTE=525  DRY
  M-E-2-HRL       Sw=0.42  TTE=148  DRY
  M-50-HRL        Sw=0.43  TTE=379  WET
  M-11-HRL        Sw=0.45  TTE=409  WET

Spearman ρ(Sw, TTE) = 0.133, p = 0.6236

→ POSITIVE correlation: higher Sw → longer TTE.
  This is the WRONG physics sign. Higher Sw should mean FASTER breakthrough.
  Interpretation: likely a VINTAGE CONFOUNDER. Older wells drilled higher in the
  gas column (lower Sw) have had more time for depletion-driven water rise,
  while ne

## 3. Vintage Analysis

In [5]:
# Spud date vs TTE
df_tte['spud_ordinal'] = df_tte['first_prod_date'].map(lambda d: d.toordinal())

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in df_tte.iterrows():
    color = COLOR_WET if row['bt_detected'] else COLOR_DRY
    if row['is_horizontal']: color = '#9467bd'
    ax.scatter(row['first_prod_date'], row['tte_months'], c=color, s=80, zorder=3)
    label = row['well'].replace('M-','').replace('-HRL','')
    ax.annotate(label, (row['first_prod_date'], row['tte_months']),
               fontsize=7, xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('First Production Date')
ax.set_ylabel('Time to Event (months)')
ax.set_title('Vintage Analysis: Spud Date vs Time to Event')
ax.grid(alpha=0.3)

# Correlation
rho_spud, p_spud = sp_stats.spearmanr(df_tte['spud_ordinal'], df_tte['tte_months'])
ax.text(0.05, 0.95, f'ρ(spud, TTE)={rho_spud:.2f}, p={p_spud:.3f}',
       transform=ax.transAxes, fontsize=9, va='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_vintage_vs_tte.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved 04_vintage_vs_tte.png')

# Compare spud correlation vs static property correlations
print(f'\nVintage analysis — Spearman correlations with TTE:')
print(f'  Spud date:        ρ = {rho_spud:.3f} (p={p_spud:.4f})')

for feat in feats_tte:
    valid = df_tte[[feat, 'tte_months']].dropna()
    if len(valid) >= 5:
        r, p = sp_stats.spearmanr(valid[feat], valid['tte_months'])
        print(f'  {feat:<20} ρ = {r:.3f} (p={p:.4f})')

print('\n→ If |ρ(spud)| > |ρ(feature)| for all features, vintage is a stronger predictor.')
print('  This would indicate confounding: apparent feature effects may just be time effects.')

Saved 04_vintage_vs_tte.png

Vintage analysis — Spearman correlations with TTE:
  Spud date:        ρ = -0.860 (p=0.0000)
  porosity             ρ = -0.309 (p=0.2278)
  permeability_md      ρ = -0.001 (p=0.9963)
  sw                   ρ = -0.057 (p=0.8292)
  net_pay_m            ρ = -0.182 (p=0.4846)
  perf_midpoint_md     ρ = -0.690 (p=0.0022)
  chlorides_ppm        ρ = 0.374 (p=0.1387)
  gas_gravity          ρ = -0.467 (p=0.0588)

→ If |ρ(spud)| > |ρ(feature)| for all features, vintage is a stronger predictor.
  This would indicate confounding: apparent feature effects may just be time effects.


## 4. Perforation Depth vs Sw

In [6]:
# Check for capillary transition zone behavior
df_vert_only = df[~df['is_horizontal'] & ~df['well'].isin(EXCLUDED_WELLS)].copy()

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in df_vert_only.iterrows():
    color = COLOR_WET if row['bt_detected'] else COLOR_DRY
    ax.scatter(row['perf_midpoint_md'], row['sw'], c=color, s=80, zorder=3)
    label = row['well'].replace('M-','').replace('-HRL','')
    ax.annotate(label, (row['perf_midpoint_md'], row['sw']),
               fontsize=7, xytext=(5, 3), textcoords='offset points')

ax.set_xlabel('Perforation Midpoint (m MD)')
ax.set_ylabel('Water Saturation (Sw)')
ax.set_title('Perf Depth vs Sw — Capillary Transition Zone Check')
ax.grid(alpha=0.3)

rho, p = sp_stats.spearmanr(df_vert_only['perf_midpoint_md'], df_vert_only['sw'])
ax.text(0.05, 0.95, f'ρ={rho:.2f}, p={p:.3f}', transform=ax.transAxes,
       fontsize=9, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

print(f'Perf midpoint vs Sw: ρ={rho:.3f}, p={p:.4f}')
if rho > 0:
    print('Positive correlation: deeper perfs → higher Sw. Consistent with capillary transition zone.')
    print('Wells perforated closer to GWC see higher initial Sw.')
else:
    print('No clear transition zone signal in the data.')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_sw_vs_perf_depth.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 04_sw_vs_perf_depth.png')

Perf midpoint vs Sw: ρ=0.001, p=0.9957
Positive correlation: deeper perfs → higher Sw. Consistent with capillary transition zone.
Wells perforated closer to GWC see higher initial Sw.
Saved 04_sw_vs_perf_depth.png


## 5. Correlation Matrices

In [7]:
# Pearson and Spearman for all static features + event time
corr_feats = ['porosity', 'permeability_md', 'sw', 'net_pay_m', 'chlorides_ppm',
              'perf_midpoint_md', 'gas_gravity', 'tte_months']

# Verticals only (horizontals have different net_pay meaning)
df_corr = df_vert_only.copy()
df_corr['tte_months'] = df_corr.apply(
    lambda r: r['bt_month_index'] if r['bt_detected'] else r['production_months'], axis=1
).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
pearson = df_corr[corr_feats].corr(method='pearson')
sns.heatmap(pearson, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[0],
            vmin=-1, vmax=1, square=True)
axes[0].set_title('Pearson Correlation')

# Spearman
spearman = df_corr[corr_feats].corr(method='spearman')
sns.heatmap(spearman, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1],
            vmin=-1, vmax=1, square=True)
axes[1].set_title('Spearman Correlation')

plt.tight_layout()
plt.savefig(FIGURES_DIR / '04_correlation_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved 04_correlation_matrices.png')

# Flag redundant pairs
print('\nRedundant pairs (|ρ| > 0.8):')
for i in range(len(corr_feats)):
    for j in range(i+1, len(corr_feats)):
        r_p = pearson.iloc[i, j]
        r_s = spearman.iloc[i, j]
        if abs(r_p) > 0.8 or abs(r_s) > 0.8:
            print(f'  {corr_feats[i]} vs {corr_feats[j]}: Pearson={r_p:.2f}, Spearman={r_s:.2f}')

print('\nNote: correlation ≠ causation. High Pearson does not mean useful for prediction;')
print('low Pearson does not mean useless (nonlinear relationships, confounders).')

Saved 04_correlation_matrices.png

Redundant pairs (|ρ| > 0.8):

Note: correlation ≠ causation. High Pearson does not mean useful for prediction;
low Pearson does not mean useless (nonlinear relationships, confounders).


## Chlorides Analysis — With and Without M-123H

In [8]:
# Check if M-123H chlorides outlier drives spurious correlations
print('Chlorides vs TTE (all wells):')
valid = df_corr[['chlorides_ppm', 'tte_months']].dropna()
r_all, p_all = sp_stats.spearmanr(valid['chlorides_ppm'], valid['tte_months'])
print(f'  ρ = {r_all:.3f}, p = {p_all:.4f}')

print('\nChlorides vs TTE (excluding M-123H):')
df_no123 = df_corr[df_corr['well'] != 'M-123H-HRL']
valid2 = df_no123[['chlorides_ppm', 'tte_months']].dropna()
if len(valid2) >= 5:
    r_ex, p_ex = sp_stats.spearmanr(valid2['chlorides_ppm'], valid2['tte_months'])
    print(f'  ρ = {r_ex:.3f}, p = {p_ex:.4f}')
    if abs(r_all - r_ex) > 0.15:
        print(f'  → M-123H substantially changes the correlation (Δρ = {abs(r_all-r_ex):.3f}).')
        print('    This is a single-point outlier effect. Chlorides conclusions should note this.')
    else:
        print(f'  → M-123H has modest influence (Δρ = {abs(r_all-r_ex):.3f}).')

Chlorides vs TTE (all wells):
  ρ = 0.299, p = 0.2602

Chlorides vs TTE (excluding M-123H):
  ρ = 0.299, p = 0.2602
  → M-123H has modest influence (Δρ = 0.000).


## Findings

1. **Sw wrong sign**: If observed — wells with LOW Sw broke through faster than wells with HIGH Sw. This is a vintage confounder, not a physics signal. Older wells were drilled higher in the gas column (low Sw) and have had centuries of depletion time.
2. **Vintage is a strong predictor**: Spud date likely correlates more strongly with TTE than any single static property, confirming that time-on-production dominates over rock properties.
3. **Perforation depth vs Sw**: Checks for capillary transition zone behavior — deeper perfs near GWC should have higher Sw.
4. **Analog-copied horizontals**: Appear as identical points in scatter plots, confirming they are not independent observations.
5. **Chlorides**: M-123H outlier (1215 ppm) may drive spurious correlations. Documented with/without analysis.
6. **Correlation ≠ causation**: High Pearson between features does not mean one causes the other or that either is useful for prediction.
7. **Permeability and porosity**: Moderate positive correlation expected from Kozeny-Carman.
8. **Net pay**: Cannot be used in cross-well comparison when mixing verticals and horizontals.
9. **Gas gravity**: Tight range (0.69–0.80) — unlikely to be a strong discriminator.
10. **Redundant pairs** flagged for VIF analysis in Phase 6.